# Recording subset selection

The extend run in `~/Desktop/input_large.yaml` embedded 37 Drive folders (~1,640 recordings, 1985–2025).
That is too much to download for the noise collection and the band check, so this notebook picks a
**small, representative subset** and downloads only that.

**How it chooses** (no audio needed; it uses Drive metadata and the hits in `results.csv`):

1. **Strata = era × device.** The recordings span 40 years of very different gear (SonyUW housings, Hi8,
   VHS-C, DV, GoPro rigs …), which differ in bandwidth, tape hiss and motor noise. Every stratum gets a
   quota proportional to √(its size), at least `MIN_PER_STRATUM`, so rare gear is covered without the big
   strata dominating.
2. **Roles inside a stratum**, filled in this order:
   - `known_whistle`: most hits in the clean whistle clusters from the UI review (for the band check),
   - `whistle_rich`: most short (< 1.5 s) WHISTLE hits per hour,
   - `background`: highest share of long (> 2 s) hits, i.e. what the model matched on background
     (hard negatives for the noise set),
   - `random`: the rest, seeded, so the noise sample stays unbiased.
   Plus `pvl`: a few recordings from the folders that have a PVL shotlog (annotated encounters).
3. **Budget:** files above `MAX_FILE_GB` are skipped, and random picks are dropped from the largest strata
   until the total is under `BUDGET_GB`.
4. **Hold-out by recording:** ~`HOLDOUT_FRAC` of the picks in every stratum are marked `holdout` and must
   not be used to build the noise set or to tune anything.

Output: `manifest.csv` in `OUT_DIR` (one row per selected recording, with its role, stratum and split), then
an optional download into the same folder. Rerunning is safe: the Drive listing is cached and finished
downloads are skipped.

**Auth:** the same environment variables as the RunPod handler (`GOOGLE_REFRESH_TOKEN` +
`GOOGLE_CLIENT_ID` + `GOOGLE_CLIENT_SECRET`, or `GOOGLE_SERVICE_ACCOUNT_JSON`), via `webui/gdrive_helper.py`.

In [ ]:
import json, os, re, sys, threading
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd

REPO = Path.cwd().parent if Path.cwd().name == "analysis_notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "webui"))

# ---- configuration ----
INPUT_YAML   = Path.home() / "Desktop/input_large.yaml"          # the extend job input (JSON despite the name)
RESULTS_CSV  = Path.home() / "Documents/dolphin_web_data/results.csv"
OUT_DIR      = Path.home() / "Documents/audio/subset"            # manifest + downloaded audio
LISTING_CSV  = OUT_DIR / "drive_listing.csv"                     # cached Drive listing

BUDGET_GB        = 20      # total download size
MAX_FILE_GB      = 1.5     # skip single files larger than this
TARGET_RECORDINGS = 90     # before the budget cut
MIN_PER_STRATUM  = 2
N_PVL            = 6       # extra picks from folders with a PVL shotlog
HOLDOUT_FRAC     = 0.15
SEED             = 0

SHORT_S, LONG_S = 1.5, 2.0                                       # hit-length cut-offs from the UI review
MIN_HITS_FOR_ROLE = 20     # whistle_rich / background picks need at least this many hits (rates on 2 hits are noise)
CLEAN_WHISTLE_CLUSTERS = [431, 128, 460, 89, 331, 420, 440, 418, 358, 202, 386]

OUT_DIR.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(SEED)

## Folders and Drive listing

In [ ]:
job = json.loads(INPUT_YAML.read_text())["input"]
folders = job["gdrive_folder_id"]
pvl = job.get("pvl_file_id") or [None] * len(folders)
pvl = pvl if isinstance(pvl, list) else [pvl] * len(folders)
folder_info = pd.DataFrame({"folder_id": folders, "folder_idx": range(len(folders)),
                            "has_pvl": [p is not None for p in pvl]})
print(f"{len(folders)} folders, {folder_info.has_pvl.sum()} with a PVL shotlog")

In [ ]:
AUDIO_EXT = (".wav", ".m4a")

def list_folder(service, folder_id):
    # direct children only, like the handler's list_audio_files()
    files, token = [], None
    while True:
        res = service.files().list(
            q=f"'{folder_id}' in parents and trashed = false",
            fields="nextPageToken, files(id, name, size, mimeType)", pageSize=1000, pageToken=token,
            supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
        files += [f for f in res.get("files", []) if f["name"].lower().endswith(AUDIO_EXT)]
        token = res.get("nextPageToken")
        if not token:
            return files

if LISTING_CSV.exists():
    listing = pd.read_csv(LISTING_CSV)
    print(f"cached listing: {len(listing)} files ({LISTING_CSV})")
else:
    from gdrive_helper import get_drive_service
    service = get_drive_service()
    rows = []
    for i, fid in enumerate(folders):
        files = list_folder(service, fid)
        rows += [{"folder_id": fid, "file_id": f["id"], "name": f["name"], "size": int(f.get("size", 0))} for f in files]
        print(f"[{i + 1}/{len(folders)}] {fid}: {len(files)} audio files")
    listing = pd.DataFrame(rows)
    listing.to_csv(LISTING_CSV, index=False)
print(f"total: {len(listing)} files, {listing['size'].sum() / 1e9:.1f} GB")

## Metadata per recording

Era and device come from the file name (`MMDDYY..._<device>`). Hit statistics come from the extend run's
`results.csv` (NOISE-classified hits are already dropped there).

In [ ]:
def year_of(stem):
    m = re.match(r"(\d{2})(\d{2})(\d{2})", stem)
    if not m:
        return np.nan
    yy = int(m.group(3))
    return 1900 + yy if yy > 50 else 2000 + yy

DEVICE_FAMILIES = [  # (regex on the lower-cased name after the date, family)
    (r"gopro|manta|skate|eagle|tascam", "modern rig"),
    (r"sonyuw", "SonyUW"),
    (r"^reg", "Reg (PCA/DV)"),
    (r"hi8", "Hi8"),
    (r"vhs", "VHS-C"),
    (r"^xr", "XR"),
    (r"^pc[bc]", "PC (B/C)"),
    (r"canon", "Canon"),
]

def device_of(stem):
    rest = re.sub(r"^\d+[ _]*", "", stem).lower()
    for pat, fam in DEVICE_FAMILIES:
        if re.search(pat, rest):
            return fam
    return "other"

def era_of(year):
    if np.isnan(year):
        return "unknown"
    return f"{int(year // 10 * 10)}s"

rec = listing.copy()
rec["stem"] = rec.name.str.replace(r"\.(wav|m4a)$", "", regex=True, flags=re.I)
rec = rec.drop_duplicates("stem")  # same stem in several folders: keep the first, like the handler
rec = rec.merge(folder_info, on="folder_id")
rec["year"] = rec.stem.map(year_of)
rec["era"] = rec.year.map(era_of)
rec["device"] = rec.stem.map(device_of)
rec["stratum"] = rec.era + " · " + rec.device
rec["gb"] = rec["size"] / 1e9

hits = pd.read_csv(RESULTS_CSV, usecols=["stem", "submodel_id", "classification", "start_time_s", "stop_time_s"])
hits["dur"] = hits.stop_time_s - hits.start_time_s
hits["short_whistle"] = (hits.classification == "WHISTLE") & (hits.dur < SHORT_S)
hits["long"] = hits.dur > LONG_S
hits["clean_whistle"] = hits.submodel_id.isin(CLEAN_WHISTLE_CLUSTERS)
g = hits.groupby("stem")
stats = pd.DataFrame({
    "n_hits": g.size(),
    "approx_hours": g.stop_time_s.max() / 3600,                      # last hit end: a lower bound on length
    "short_whistles": g.short_whistle.sum(),
    "long_share": g.long.mean(),
    "clean_whistle_hits": g.clean_whistle.sum(),
})
stats["whistles_per_hour"] = stats.short_whistles / stats.approx_hours.clip(lower=0.05)
rec = rec.merge(stats, left_on="stem", right_index=True, how="left").fillna(
    {"n_hits": 0, "short_whistles": 0, "long_share": 0, "clean_whistle_hits": 0, "whistles_per_hour": 0})
print(f"{len(rec)} recordings, {rec.n_hits.gt(0).sum()} with hits in results.csv")
rec.groupby("stratum").agg(recordings=("stem", "size"), GB=("gb", "sum"),
                           short_whistles=("short_whistles", "sum")).sort_values("recordings", ascending=False).round(1)

## Selection

In [ ]:
eligible = rec[rec.gb <= MAX_FILE_GB].copy()
print(f"skipping {len(rec) - len(eligible)} files over {MAX_FILE_GB} GB")

sizes = eligible.groupby("stratum").size()
weight = np.sqrt(sizes)
quota = np.maximum(MIN_PER_STRATUM, np.round(TARGET_RECORDINGS * weight / weight.sum())).astype(int)
quota = np.minimum(quota, sizes)

ROLE_ORDER = [  # (role, sort column, minimum value to qualify)
    ("known_whistle", "clean_whistle_hits", 1),
    ("whistle_rich", "whistles_per_hour", 1e-9),
    ("background", "long_share", 1e-9),
]

picks = []
for stratum, q in quota.items():
    pool = eligible[eligible.stratum == stratum]
    chosen = []
    for role, col, minimum in ROLE_ORDER:
        if len(chosen) >= q:
            break
        cand = pool[~pool.stem.isin(chosen) & (pool[col] >= minimum)]
        if role != "known_whistle":
            cand = cand[cand.n_hits >= MIN_HITS_FOR_ROLE]
        cand = cand.sort_values(col, ascending=False)
        if len(cand):
            chosen.append(cand.stem.iloc[0]); picks.append((cand.stem.iloc[0], role))
    rest = pool[~pool.stem.isin(chosen)]
    n_random = min(q - len(chosen), len(rest))
    for stem in rest.stem.sample(n_random, random_state=SEED):
        chosen.append(stem); picks.append((stem, "random"))

# annotated encounters: a few extra recordings from PVL folders, preferring ones with whistles
taken = {s for s, _ in picks}
pvl_pool = eligible[eligible.has_pvl & ~eligible.stem.isin(taken)].sort_values("whistles_per_hour", ascending=False)
picks += [(s, "pvl") for s in pvl_pool.stem.head(N_PVL)]

sel = eligible.set_index("stem").loc[[s for s, _ in picks]].reset_index()
sel["role"] = [r for _, r in picks]
print(f"before budget: {len(sel)} recordings, {sel.gb.sum():.1f} GB")

In [ ]:
# Budget: drop random picks (largest file first) from the stratum with the most picks, keep >= 1 per stratum
while sel.gb.sum() > BUDGET_GB:
    counts = sel.stratum.value_counts()
    droppable = sel[(sel.role == "random") & sel.stratum.map(counts).gt(1)]
    if droppable.empty:
        print("WARNING: only role picks left and still over budget; raise BUDGET_GB or lower MAX_FILE_GB")
        break
    n_in_stratum = droppable.stratum.map(counts)
    worst = droppable[n_in_stratum == n_in_stratum.max()]
    sel = sel.drop(worst.gb.idxmax())

# Hold-out by recording: in every stratum, every k-th pick (random ones first) goes to holdout
k = max(2, round(1 / HOLDOUT_FRAC))
sel["split"] = "dev"
for stratum, grp in sel.groupby("stratum"):
    order = grp.assign(r=grp.role.ne("random")).sort_values(["r", "stem"]).index  # random picks first
    sel.loc[order[k - 1::k], "split"] = "holdout"
    if len(grp) >= 3 and not (sel.loc[grp.index, "split"] == "holdout").any():
        sel.loc[order[0], "split"] = "holdout"

print(f"selected {len(sel)} recordings, {sel.gb.sum():.1f} GB "
      f"({sel.gb.sum() / rec.gb.sum():.1%} of {rec.gb.sum():.0f} GB), "
      f"{sel.stratum.nunique()} of {rec.stratum.nunique()} strata, holdout {sel.split.eq('holdout').sum()}")
print(sel.role.value_counts().to_string())

## Coverage check

In [ ]:
cov = pd.DataFrame({
    "all_recordings": rec.groupby("stratum").size(),
    "selected": sel.groupby("stratum").size(),
    "holdout": sel[sel.split == "holdout"].groupby("stratum").size(),
    "selected_GB": sel.groupby("stratum").gb.sum().round(2),
}).fillna(0).astype({"selected": int, "holdout": int}).sort_values("all_recordings", ascending=False)
print("strata with no pick:", list(cov.index[cov.selected == 0]) or "none")
cov

In [ ]:
# Do the picks look like the whole collection on the hit statistics? (random picks should)
cols = ["whistles_per_hour", "long_share", "n_hits"]
pd.concat({"all": rec[cols].describe().T[["25%", "50%", "75%"]],
           "selected": sel[cols].describe().T[["25%", "50%", "75%"]],
           "random picks": sel[sel.role == "random"][cols].describe().T[["25%", "50%", "75%"]]}, axis=1).round(2)

In [ ]:
manifest = sel[["stem", "name", "file_id", "folder_id", "gb", "year", "era", "device", "stratum", "role", "split",
                "has_pvl", "n_hits", "short_whistles", "whistles_per_hour", "long_share", "clean_whistle_hits"]]
manifest.to_csv(OUT_DIR / "manifest.csv", index=False)
print(f"wrote {OUT_DIR / 'manifest.csv'}")
manifest.head(10)

## Download

Set `DOWNLOAD = True` to fetch the manifest into `OUT_DIR`. Files land as `<name>`; a download goes to
`<name>.part` first and is renamed when complete, and existing files are skipped, so this can be rerun.

In [ ]:
DOWNLOAD = False
WORKERS = 4

if DOWNLOAD:
    from gdrive_helper import download, thread_service

    def fetch(row):
        path = OUT_DIR / row.name
        if path.exists():
            return "skipped"
        tmp = path.with_suffix(path.suffix + ".part")
        download(thread_service(), row.file_id, tmp)
        tmp.rename(path)
        return "downloaded"

    rows = list(manifest.itertuples(index=False))
    done = 0
    with ThreadPoolExecutor(WORKERS) as pool:
        for status in pool.map(fetch, rows):
            done += 1
            if done % 10 == 0 or done == len(rows):
                print(f"  {done}/{len(rows)}")
    have = sum((OUT_DIR / n).exists() for n in manifest.name)
    print(f"{have}/{len(manifest)} files in {OUT_DIR}")